# v3 BO Hyperparameter Correlation Analysis

Run `04_pareto_efficiency_analysis_v3.ipynb` first.

Supports single- or multi-run configs (`runs` in `configs/selection_notebooks.yaml`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler

try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

cwd = Path.cwd().resolve()
project_root = cwd
for _ in range(6):
    if (project_root / "configs").is_dir() and (project_root / "src").is_dir():
        break
    project_root = project_root.parent
else:
    raise RuntimeError("Could not find project root")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.common.config import load_config, resolve_path
from src.legacy.stage04_selection.pareto_analysis import analyze_hyperparameters
from src.stage04_eval_select.notebook_io import (
    STRATEGY_FILES,
    ensure_run_dirs,
    load_top_k_sets,
    resolve_comparison_dirs,
    resolve_run_dirs,
    resolve_runs,
)

nb_cfg = load_config(project_root / "configs" / "selection_notebooks.yaml")
runs = resolve_runs(nb_cfg)
run_dirs = {run["run_id"]: resolve_run_dirs(run, project_root) for run in runs}
for dirs in run_dirs.values():
    ensure_run_dirs(dirs)
cmp_dirs = resolve_comparison_dirs(nb_cfg, project_root)
multi_run = len(runs) > 1

primary_run = runs[0]
figures_dir = run_dirs[primary_run["run_id"]]["figures_dir"]
tables_dir = run_dirs[primary_run["run_id"]]["tables_dir"]
top_models_dir = run_dirs[primary_run["run_id"]]["top_models_dir"]
HP_COLS = nb_cfg["hyperparameters"]
sns.set_style("whitegrid")
print(f"Runs: {[r['run_id'] for r in runs]} ({'compare' if multi_run else 'single'} mode)"

## 1. Load top-k sets

In [ ]:
top_sets = load_top_k_sets(runs, project_root)
for (run_id, strategy), df in top_sets.items():
    label = df["model_label"].iloc[0] if len(df) else run_id
    print(f"{label} / {strategy}: {len(df)} trials")

## 2. Correlation + Cohen's d

In [ ]:
def metrics_for(name, df):
    if name == "eval_select":
        return [c for c in nb_cfg["performance_metrics_v3"] if c in df.columns]
    return [c for c in nb_cfg["performance_metrics_legacy"] if c in df.columns]

corr_parts = []
for (run_id, name), df in top_sets.items():
    corr = analyze_hyperparameters(df, HP_COLS, metrics_for(name, df))
    corr["run_id"] = run_id
    corr["model_label"] = df["model_label"].iloc[0] if len(df) else run_id
    corr["strategy"] = name
    corr_parts.append(corr)
    path = run_dirs[run_id]["tables_dir"] / f"correlation_analysis_{name}.csv"
    corr.to_csv(path, index=False)
    print("Saved", path)

corr_all = pd.concat(corr_parts, ignore_index=True)
if cmp_dirs and multi_run:
    cmp_path = cmp_dirs["tables_dir"] / "correlation_analysis_all_runs.csv"
    corr_all.to_csv(cmp_path, index=False)
    print("Saved", cmp_path)

## 3. Boxplots

In [ ]:
parts = []
for (run_id, name), df in top_sets.items():
    scaler = MinMaxScaler()
    norm = df.copy()
    norm[HP_COLS] = scaler.fit_transform(df[HP_COLS])
    part = norm.melt(value_vars=HP_COLS, var_name="Hyperparameter", value_name="Value")
    part["Strategy"] = name
    part["model_label"] = df["model_label"].iloc[0] if len(df) else run_id
    parts.append(part)
melted = pd.concat(parts, ignore_index=True)

for run in runs:
    rid = run["run_id"]
    label = run.get("label", rid)
    sub = melted[melted["model_label"] == label]
    fig, ax = plt.subplots(figsize=(14, 7))
    sns.boxplot(data=sub, x="Value", y="Hyperparameter", hue="Strategy", orient="h", ax=ax)
    ax.set_title(f"Hyperparameter distributions — {label}")
    plt.tight_layout()
    out = run_dirs[rid]["figures_dir"] / "hyperparameter_boxplots_all_strategies.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved {out}")
    plt.show()

if multi_run and cmp_dirs:
    fig, ax = plt.subplots(figsize=(16, 8))
    sns.boxplot(
        data=melted[melted["Strategy"] == "eval_select"],
        x="Value", y="Hyperparameter", hue="model_label", orient="h", ax=ax,
    )
    ax.set_title("eval_select top-k hyperparameters — all runs")
    plt.tight_layout()
    out = cmp_dirs["figures_dir"] / "hyperparameter_boxplots_eval_select_by_run.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved {out}")
    plt.show()

## 4. OLS + VIF (eval_select)

In [ ]:
if HAS_STATSMODELS:
    ols_all = []
    for run in runs:
        rid = run["run_id"]
        label = run.get("label", rid)
        df_primary = top_sets[(rid, "eval_select")]
        X = df_primary[HP_COLS].astype(float)
        X_const = sm.add_constant(X)
        ols_rows = []
        for metric in [c for c in ["coherence_c_v", "topic_diversity", "n_topics"] if c in df_primary.columns]:
            model = sm.OLS(df_primary[metric].astype(float), X_const).fit()
            for param in X_const.columns:
                row = {
                    "run_id": rid, "model_label": label, "metric": metric,
                    "parameter": param, "coef": model.params[param], "pvalue": model.pvalues[param],
                }
                ols_rows.append(row)
                ols_all.append(row)
            vif = pd.DataFrame([
                {"run_id": rid, "model_label": label, "metric": metric, "parameter": col,
                 "vif": variance_inflation_factor(X.values, i)}
                for i, col in enumerate(X.columns)
            ])
            vif.to_csv(run_dirs[rid]["tables_dir"] / f"vif_eval_select_{metric}.csv", index=False)
        pd.DataFrame(ols_rows).to_csv(run_dirs[rid]["tables_dir"] / "ols_eval_select.csv", index=False)
    if cmp_dirs and multi_run:
        pd.DataFrame(ols_all).to_csv(cmp_dirs["tables_dir"] / "ols_eval_select_all_runs.csv", index=False)
    print("Saved OLS/VIF per run")
else:
    print("statsmodels not installed")

## 5. Random Forest importance

In [ ]:
rf_rows = []
for (run_id, name), df in top_sets.items():
    label = df["model_label"].iloc[0] if len(df) else run_id
    metrics = [c for c in ["coherence_c_v", "topic_diversity", "n_topics", "Combined_Score", "weighted_score"] if c in df.columns]
    X = df[HP_COLS].astype(float)
    for metric in metrics:
        if len(df) < 3:
            continue
        rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(X, df[metric].astype(float))
        for hp, imp in zip(HP_COLS, rf.feature_importances_):
            rf_rows.append({
                "run_id": run_id, "model_label": label, "strategy": name,
                "metric": metric, "hyperparameter": hp, "importance": imp,
            })
rf_df = pd.DataFrame(rf_rows)
for run in runs:
    rid = run["run_id"]
    sub = rf_df[rf_df["run_id"] == rid]
    sub.to_csv(run_dirs[rid]["tables_dir"] / "random_forest_importance.csv", index=False)
if cmp_dirs and multi_run:
    rf_df.to_csv(cmp_dirs["tables_dir"] / "random_forest_importance_all_runs.csv", index=False)
    pivot = rf_df[rf_df["strategy"] == "eval_select"].pivot_table(
        index=["hyperparameter", "metric"], columns="model_label", values="importance"
    )
    pivot.to_csv(cmp_dirs["tables_dir"] / "random_forest_importance_eval_select_pivot.csv")
    fig, ax = plt.subplots(figsize=(10, 6))
    sub = rf_df[(rf_df["strategy"] == "eval_select") & (rf_df["metric"] == "coherence_c_v")]
    sns.barplot(data=sub, x="importance", y="hyperparameter", hue="model_label", orient="h", ax=ax)
    ax.set_title("RF importance (eval_select, coherence_c_v)")
    plt.tight_layout()
    plt.savefig(cmp_dirs["figures_dir"] / "random_forest_importance_coherence_by_run.png", dpi=150)
    plt.show()
print("Saved random forest importance")

## 6. XGBoost (optional)

In [ ]:
if HAS_XGB:
    xgb_rows = []
    for run in runs:
        rid = run["run_id"]
        label = run.get("label", rid)
        df_primary = top_sets[(rid, "eval_select")]
        X = df_primary[HP_COLS].astype(float)
        for metric in [c for c in ["coherence_c_v", "topic_diversity", "n_topics"] if c in df_primary.columns]:
            model = XGBRegressor(n_estimators=200, random_state=42, verbosity=0).fit(X, df_primary[metric].astype(float))
            for hp, imp in zip(HP_COLS, model.feature_importances_):
                xgb_rows.append({"run_id": rid, "model_label": label, "metric": metric, "hyperparameter": hp, "importance": imp})
        pd.DataFrame([r for r in xgb_rows if r["run_id"] == rid]).to_csv(
            run_dirs[rid]["tables_dir"] / "xgboost_importance_eval_select.csv", index=False
        )
    if cmp_dirs and multi_run:
        pd.DataFrame(xgb_rows).to_csv(cmp_dirs["tables_dir"] / "xgboost_importance_eval_select_all_runs.csv", index=False)
    print("Saved XGBoost importance")
else:
    print("xgboost not installed")